In [1]:
!pip install transformers torch scikit-learn pandas numpy tqdm

Defaulting to user installation because normal site-packages is not writeable


In [2]:
!pip install --upgrade transformers

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import BertTokenizer
from transformers import BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset

[HAMI-core Msg(923:139683116907840:libvgpu.c:839)]: Initializing.....
/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[HAMI-core Msg(923:139683116907840:libvgpu.c:855)]: Initialized


In [17]:
df = pd.read_csv("deceptive-opinion.csv")
df.columns = ['deceptive', 'hotel', 'polarity', 'source', 'review']
df = df.rename(columns={
    "text": "review",
    "deceptive": "label"
})

df["label"] = df["label"].map({
    "truthful": 0,
    "deceptive": 1
})
df.head()

,label,hotel,polarity,source,review
0,0,conrad,positive,TripAdvisor,We stayed for a one night getaway with family ...
1,0,hyatt,positive,TripAdvisor,Triple A rate with upgrade to view room was le...
2,0,hyatt,positive,TripAdvisor,This comes a little late as I'm finally catchi...
3,0,omni,positive,TripAdvisor,The Omni Chicago really delivers on all fronts...
4,0,hyatt,positive,TripAdvisor,I asked for a high floor away from the elevato...


In [18]:
import re

def clean_text(text):
    
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

#df["text"] = df["text"].apply(clean_text)
df["review"] = df["review"].apply(clean_text)

df.head()

,label,hotel,polarity,source,review
0,0,conrad,positive,TripAdvisor,we stayed for a one night getaway with family ...
1,0,hyatt,positive,TripAdvisor,triple a rate with upgrade to view room was le...
2,0,hyatt,positive,TripAdvisor,this comes a little late as im finally catchin...
3,0,omni,positive,TripAdvisor,the omni chicago really delivers on all fronts...
4,0,hyatt,positive,TripAdvisor,i asked for a high floor away from the elevato...


In [19]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["review"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

In [20]:
print(df.columns)

Index(['label', 'hotel', 'polarity', 'source', 'review'], dtype='object')


In [21]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [22]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [23]:
class ReviewDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        
        return item

    def __len__(self):
        return len(self.labels)

In [24]:
train_dataset = ReviewDataset(train_encodings, train_labels)
val_dataset = ReviewDataset(val_encodings, val_labels)

In [25]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15300.38it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider t

In [26]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
[HAMI-core Msg(1199:140249703851840:multiprocess_memory_limit.c:455)]: Calling exit handler 1199


In [14]:
import sys
!{sys.executable} -m pip install accelerate

Defaulting to user installation because normal site-packages is not writeable


In [27]:
import transformers
print(transformers.__version__)

5.3.0


In [28]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.72it/s]


TrainOutput(global_step=480, training_loss=0.4143426259358724, metrics={'train_runtime': 21.4357, 'train_samples_per_second': 179.14, 'train_steps_per_second': 22.393, 'total_flos': 505173226291200.0, 'train_loss': 0.4143426259358724, 'epoch': 3.0})

In [29]:
predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)

accuracy = accuracy_score(val_labels, preds)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.859375


In [30]:
model.save_pretrained("fake_review_model")

tokenizer.save_pretrained("fake_review_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.69it/s]


('fake_review_model/tokenizer_config.json', 'fake_review_model/tokenizer.json')

In [33]:
import torch

def predict_review(text):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    # Move inputs to device
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    predicted_class = torch.argmax(logits).item()

    if predicted_class == 1:
        return "Fake Review"
    else:
        return "Genuine Review"

In [35]:
predict_review("This hotel was fantastic and the staff were amazing")

'Fake Review'

In [36]:
reviews = [
    "Best hotel ever highly recommended",
    "Terrible stay dirty rooms and bad service",
    "Amazing experience loved the view"
]

for r in reviews:
    print(r, "->", predict_review(r))

Best hotel ever highly recommended -> Genuine Review
Terrible stay dirty rooms and bad service -> Genuine Review
Amazing experience loved the view -> Fake Review


In [37]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [38]:
def predict_review(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    prediction = torch.argmax(logits).item()

    if prediction == 1:
        return "Fake Review"
    else:
        return "Genuine Review"

In [39]:
while True:
    
    review = input("Enter your review (type 'exit' to stop): ")

    if review.lower() == "exit":
        break

    result = predict_review(review)

    print("Prediction:", result)
    print()

Enter your review (type 'exit' to stop):  This hotel was amazing and the staff were very friendly


Prediction: Fake Review



Enter your review (type 'exit' to stop):  This is the best hotel in the world highly recommended five stars


Prediction: Fake Review



Enter your review (type 'exit' to stop):  I really enjoyed the hotel and the staff we also helpful


Prediction: Fake Review



Enter your review (type 'exit' to stop):  exit
